# 1. Fresh start - 20260315 - Paper trading

# 2. Import Libraries 

- all libraries needed 
- can add new libraries with pip install in the cell. 
- better option to add them in requirements.txt and run ``` pip install -r requirements.txt```

In [1]:
import ib_async 
# import ib_insync - exclude insync to avoid conflicts. 
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup as bs
import datetime

print(f"Success! Using ib_async version: {ib_async.__version__}")
# print(f"Success! Using ib_async version: {ib_insync.__version__}")

Success! Using ib_async version: 2.1.0


# 2.1:

-> difference between 'import ib_async ' and 'from ib_async import *'
> import ib_async (The Organized Way)

> This imports the library as a single "namespace." To use anything inside it, you must prefix it with ib_async..

    > Example: ib = ib_async.IB()


Feature,import ib_async,from ib_async import *
Usage,ib_async.IB(),IB()
Clarity,High (explicit),Low (implicit)
Safety,Prevents name conflicts,Risk of overwriting functions
Best For,"Large, professional projects",Interactive Notebooks / Prototypes


## 2.1) Print directory of ib_async

In [2]:
print(dir(ib_async))

['AccountValue', 'Bag', 'BarData', 'BarDataList', 'Bond', 'BracketOrder', 'CFD', 'Client', 'ComboLeg', 'CommissionReport', 'Commodity', 'ConnectionStats', 'ContFuture', 'Contract', 'ContractDescription', 'ContractDetails', 'Crypto', 'DOMLevel', 'DeltaNeutralContract', 'DepthMktDataDescription', 'Dividends', 'Event', 'Execution', 'ExecutionCondition', 'ExecutionFilter', 'FamilyCode', 'Fill', 'FlexError', 'FlexReport', 'Forex', 'FundamentalRatios', 'Future', 'FuturesOption', 'HistogramData', 'HistoricalNews', 'HistoricalSchedule', 'HistoricalSession', 'HistoricalTick', 'HistoricalTickBidAsk', 'HistoricalTickLast', 'IB', 'IBC', 'IBDefaults', 'Index', 'LimitOrder', 'MarginCondition', 'MarketOrder', 'MktDepthData', 'MutualFund', 'NewsArticle', 'NewsBulletin', 'NewsProvider', 'NewsTick', 'Option', 'OptionChain', 'OptionComputation', 'Order', 'OrderComboLeg', 'OrderCondition', 'OrderState', 'OrderStateNumeric', 'OrderStatus', 'PercentChangeCondition', 'PnL', 'PnLSingle', 'PortfolioItem', 'Pos

# 2.2) Specific tools you use daily - Connection, Instruments,	Orders,Data Handling,	

In [3]:
from ib_async import IB, Watchdog, Stock, Option, Contract, Order, MarketOrder , LimitOrder, Trade, util, Ticker, BarData



| Imported Item | Do you need a name = Item() line? |Why? |
|--------------|--------------|-----------|
|IB | YES (ib = IB()) | It is your central connection engine.|
|Watchdog | Optional | Only if you want it to auto-reconnect for you|
|Stock/Option | As needed | Used to define the ticker you want to trade.|
|Order/LimitOrder | As needed | ,As needed,Used to define the price/size of a trade.|
|util | No | You just call util.something() directly.|
|Ticker/BarData | No | These are results sent to you by IBKR.|

# 3. Connect to IB gateway - paper trading 

3.1 Notes

``` util.startLoop() ``` # uncomment this line when in a notebook
- Since you are using ib_async, you are using a library built from the ground up to work natively with Python's modern asyncio framework.

- In Jupyter: Jupyter already has an active async loop running in the background. ib_async is designed to plug into it automatically.

    **You only need util.startLoop() if:**

- You are using the old ib_insync library (not ib_async).

- You are running code inside a Jupyter Notebook or IPython console.

- You are experiencing "Timeout" errors immediately upon trying to connect.

In [6]:
util.startLoop()

ib = IB()
# ib.connect('127.0.0.1', 7497, clientId=3) # RK - paper trading
ib.connect('127.0.0.1', 7496, clientId=2) # RK - regular trading

# ib.connect('127.0.0.1', 4002, clientId=1) 
# use above for IB gateway

<IB connected to 127.0.0.1:7496 clientId=2>

Error 10358, reqId 5: Fundamentals data is not allowed., contract: Stock(symbol='AAPL', exchange='SMART', currency='USD')
Error 10358, reqId 6: Fundamentals data is not allowed., contract: Stock(symbol='AAPL', exchange='SMART', currency='USD')
Error 10358, reqId 7: Fundamentals data is not allowed., contract: Stock(symbol='AAPL', exchange='SMART', currency='USD')
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. All data farms are connected: cafarm; hfarm; eufarmnj; cashfarm; usfuture; jfarm; usfarm.nj; usopt; usfarm; euhmds; fundfarm; ushmds; secdefil.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. All data farms are connected: cafarm; hfarm; eufarmnj; cashfarm; usfuture; jfarm; usfarm.nj; usopt; us

# 3.2 disconnect if needed to troubleshoot

In [ ]:
if 'ib' in globals():
    try:
        if ib.isConnected():
            print("Detected active connection. Disconnecting...")
            ib.disconnect()
            ib.waitOnUpdate(timeout=0.5) # Give it a moment to heartbeat out
    except NameError:
        pass

# 4. Contract (Stocks , Options, Bonds)

In [12]:
contract = Contract ( symbol = 'COUR', secType = 'STK', exchange ='SMART', currency = 'USD')
ib.qualifyContracts(contract)
contract

Contract(secType='STK', conId=479429945, symbol='COUR', exchange='SMART', primaryExchange='NYSE', currency='USD', localSymbol='COUR', tradingClass='COUR')

# 5. Stock

In [13]:
stock = Stock (symbol = 'COUR', exchange = 'SMART', currency = 'USD')
ib.qualifyContracts (stock)
stock

Stock(conId=479429945, symbol='COUR', exchange='SMART', primaryExchange='NYSE', currency='USD', localSymbol='COUR', tradingClass='COUR')

# 6. Contract Details 
- Output of stored data in variables cour and contract_details

In [15]:
cour = Stock(symbol = "COUR")
contract_details = ib.reqContractDetails(cour)

In [16]:
cour

Stock(symbol='COUR')

In [17]:
contract_details

[ContractDetails(contract=Contract(secType='STK', conId=479429945, symbol='COUR', exchange='SMART', primaryExchange='NYSE', currency='USD', localSymbol='COUR', tradingClass='COUR'), marketName='COUR', minTick=0.01, orderTypes='ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AON,AVGCOST,BASKET,BENCHPX,CASHQTY,COND,CONDORDER,DARKONLY,DARKPOLL,DAY,DEACT,DEACTDIS,DEACTEOD,DIS,DUR,GAT,GTC,GTD,GTT,HID,IBKRATS,ICE,IMB,IOC,LIT,LMT,LOC,MIDPX,MIT,MKT,MOC,MTL,NGCOMB,NODARK,NONALGO,OCA,OPG,OPGREROUT,PEGBENCH,PEGMID,POSTATS,POSTONLY,PREOPGRTH,PRICECHK,REL,REL2MID,RELPCTOFS,RPI,RTH,RTHIGNOPG,SCALE,SCALEODD,SCALERST,SIZECHK,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,SWEEP,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF', validExchanges='SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,DRCTEDGE,BEX,BATS,EDGEA,BYX,IEX,EDGX,FOXRIVER,PEARL,NYSENAT,LTSE,MEMX,IBEOS,OVERNIGHT,TPLUS0,PSX,T24X', priceMagnifier=1, underConId=0, longName='COURSERA INC', contractMonth='', industry='Consumer, Non-cyclical', category='Commercial Service

In [18]:
util.df(contract_details)

,contract,marketName,minTick,orderTypes,validExchanges,priceMagnifier,underConId,longName,contractMonth,industry,...,callable,putable,coupon,convertible,maturity,issueDate,nextOptionDate,nextOptionType,nextOptionPartial,notes
0,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AO...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
1,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
2,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
3,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
4,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
5,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
6,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
7,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADJUST,ALERT,ALGOCLS,ALGOOPG,ALLO...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
8,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADJUST,ALERT,ALGOCLS,ALGOOPG,ALLO...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
9,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADJUST,ALERT,ALLOC,AVGCOST,BASKET...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,


In [19]:
df_cour_contract_details = util.df(contract_details)

In [20]:
df_cour_contract_details

,contract,marketName,minTick,orderTypes,validExchanges,priceMagnifier,underConId,longName,contractMonth,industry,...,callable,putable,coupon,convertible,maturity,issueDate,nextOptionDate,nextOptionType,nextOptionPartial,notes
0,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AO...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
1,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
2,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
3,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
4,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
5,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
6,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
7,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADJUST,ALERT,ALGOCLS,ALGOOPG,ALLO...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
8,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADJUST,ALERT,ALGOCLS,ALGOOPG,ALLO...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
9,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADJUST,ALERT,ALLOC,AVGCOST,BASKET...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,


In [21]:
df_cour_contract_details.head()

,contract,marketName,minTick,orderTypes,validExchanges,priceMagnifier,underConId,longName,contractMonth,industry,...,callable,putable,coupon,convertible,maturity,issueDate,nextOptionDate,nextOptionType,nextOptionPartial,notes
0,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AO...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
1,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
2,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
3,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
4,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,


In [22]:
df_cour_contract_details.tail()

,contract,marketName,minTick,orderTypes,validExchanges,priceMagnifier,underConId,longName,contractMonth,industry,...,callable,putable,coupon,convertible,maturity,issueDate,nextOptionDate,nextOptionType,nextOptionPartial,notes
28,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.01,"ACTIVETIM,AD,ALERT,ALGO,ALLOC,AVGCOST,BASKET,B...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
29,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.01,"AD,ALERT,ALLOC,AVGCOST,BASKET,BENCHPX,CASHQTY,...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
30,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.01,"ACTIVETIM,AD,ADJUST,ALERT,ALLOC,AON,AVGCOST,BA...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
31,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.01,"ACTIVETIM,AD,ADJUST,ALERT,ALLOC,AVGCOST,BASKET...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
32,"Contract(secType='STK', conId=500290485, symbo...",COUR,0.01,"ACTIVETIM,AD,ADJUST,ALERT,ALLOC,AVGCOST,BASKET...",MEXI,1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,


In [24]:
df_cour_contract_details.orderTypes.head()

0    ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AO...
1    ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...
2    ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...
3    ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...
4    ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...
Name: orderTypes, dtype: object

In [25]:
df_cour_contract_details.contract.head()

0    Contract(secType='STK', conId=479429945, symbo...
1    Contract(secType='STK', conId=29855061, symbol...
2    Contract(secType='STK', conId=29855061, symbol...
3    Contract(secType='STK', conId=29855061, symbol...
4    Contract(secType='STK', conId=29855061, symbol...
Name: contract, dtype: object

In [31]:
util.df(df_cour_contract_details.contract.tolist())

,secType,conId,symbol,lastTradeDateOrContractMonth,strike,right,multiplier,exchange,primaryExchange,currency,localSymbol,tradingClass,includeExpired,secIdType,secId,description,issuerId,comboLegsDescrip,comboLegs,deltaNeutralContract
0,STK,479429945,COUR,,0.0,,,SMART,NYSE,USD,COUR,COUR,False,,,,,,[],None
1,STK,29855061,COUR,,0.0,,,SMART,SBF,EUR,COUR,COUR,False,,,,,,[],None
2,STK,29855061,COUR,,0.0,,,SBF,SBF,EUR,COUR,COUR,False,,,,,,[],None
3,STK,29855061,COUR,,0.0,,,CHIXEN,SBF,EUR,COURp,COUR,False,,,,,,[],None
4,STK,29855061,COUR,,0.0,,,BATEEN,SBF,EUR,COURp,COUR,False,,,,,,[],None
5,STK,29855061,COUR,,0.0,,,DXEEN,SBF,EUR,COURp,COUR,False,,,,,,[],None
6,STK,29855061,COUR,,0.0,,,AQEUEN,SBF,EUR,COURp,COUR,False,,,,,,[],None
7,STK,479429945,COUR,,0.0,,,AMEX,NYSE,USD,COUR,COUR,False,,,,,,[],None
8,STK,479429945,COUR,,0.0,,,NYSE,NYSE,USD,COUR,COUR,False,,,,,,[],None
9,STK,479429945,COUR,,0.0,,,CBOE,NYSE,USD,COUR,COUR,False,,,,,,[],None


In [32]:
df_cour_contract_details['validExchanges_list'] = ( 
    df_cour_contract_details['validExchanges'] 
        .str.split(',') 
        .apply(lambda lst: [x.strip() for x in lst]) 
)

In [33]:
df_cour_contract_details['validExchanges_list']

0     [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
1           [SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]
2           [SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]
3           [SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]
4           [SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]
5           [SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]
6           [SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]
7     [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
8     [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
9     [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
10    [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
11    [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
12    [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
13    [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
14    [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
15    [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
16    [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, ARCA...
17    [SMART, AMEX, NYSE, CBOE, PHLX, ISE, CHX, 

In [39]:
df_split = df_cour_contract_details['validExchanges_list'].apply(pd.Series)

In [40]:
df_split

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,DRCTEDGE,...,FOXRIVER,PEARL,NYSENAT,LTSE,MEMX,IBEOS,OVERNIGHT,TPLUS0,PSX,T24X
1,SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,DRCTEDGE,...,FOXRIVER,PEARL,NYSENAT,LTSE,MEMX,IBEOS,OVERNIGHT,TPLUS0,PSX,T24X
8,SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,DRCTEDGE,...,FOXRIVER,PEARL,NYSENAT,LTSE,MEMX,IBEOS,OVERNIGHT,TPLUS0,PSX,T24X
9,SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,DRCTEDGE,...,FOXRIVER,PEARL,NYSENAT,LTSE,MEMX,IBEOS,OVERNIGHT,TPLUS0,PSX,T24X


In [38]:
df_cour_contract_details.iloc[1:6]

,contract,marketName,minTick,orderTypes,validExchanges,priceMagnifier,underConId,longName,contractMonth,industry,...,putable,coupon,convertible,maturity,issueDate,nextOptionDate,nextOptionType,nextOptionPartial,notes,validExchanges_list
1,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,0,False,,,,,False,,"[SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]"
2,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,0,False,,,,,False,,"[SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]"
3,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,0,False,,,,,False,,"[SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]"
4,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,0,False,,,,,False,,"[SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]"
5,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,0,False,,,,,False,,"[SMART, SBF, CHIXEN, BATEEN, DXEEN, AQEUEN]"


In [30]:
df_cour_contract_details.iloc[0:6]

,contract,marketName,minTick,orderTypes,validExchanges,priceMagnifier,underConId,longName,contractMonth,industry,...,callable,putable,coupon,convertible,maturity,issueDate,nextOptionDate,nextOptionType,nextOptionPartial,notes
0,"Contract(secType='STK', conId=479429945, symbo...",COUR,0.0100,"ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AO...","SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,...",1,0,COURSERA INC,,"Consumer, Non-cyclical",...,False,False,0,False,,,,,False,
1,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
2,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
3,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
4,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,
5,"Contract(secType='STK', conId=29855061, symbol...",COUR,0.0005,"ADJUST,ALERT,ALLOC,BASKET,COND,CONDORDER,DAY,D...","SMART,SBF,CHIXEN,BATEEN,DXEEN,AQEUEN",1,0,COURTOIS-R,,Financial,...,False,False,0,False,,,,,False,


In [20]:
stock_NVDA = Stock ('NVDA', 'smart', 'USD')

bars = ib.reqHistoricalData(
    stock, endDateTime='', durationStr='30 D',
    barSizeSetting='1 hour', whatToShow='MIDPOINT', useRTH=True)

# convert to pandas dataframe (pandas needs to be installed):
df_NVDA = util.df(bars)
print(df_NVDA)

                         date  open  high   low  close  volume  average  \
0   2026-01-30 09:30:00-05:00  6.13  6.17  6.02   6.04    -1.0     -1.0   
1   2026-01-30 10:00:00-05:00  6.04  6.15  6.04   6.11    -1.0     -1.0   
2   2026-01-30 11:00:00-05:00  6.11  6.11  6.03   6.06    -1.0     -1.0   
3   2026-01-30 12:00:00-05:00  6.06  6.09  6.04   6.08    -1.0     -1.0   
4   2026-01-30 13:00:00-05:00  6.08  6.10  6.05   6.06    -1.0     -1.0   
..                        ...   ...   ...   ...    ...     ...      ...   
205 2026-03-13 11:00:00-04:00  6.07  6.10  6.01   6.06    -1.0     -1.0   
206 2026-03-13 12:00:00-04:00  6.06  6.08  6.00   6.04    -1.0     -1.0   
207 2026-03-13 13:00:00-04:00  6.04  6.09  6.02   6.04    -1.0     -1.0   
208 2026-03-13 14:00:00-04:00  6.04  6.08  6.02   6.08    -1.0     -1.0   
209 2026-03-13 15:00:00-04:00  6.08  6.09  6.01   6.04    -1.0     -1.0   

     barCount  
0          -1  
1          -1  
2          -1  
3          -1  
4          -1  
.. 

In [21]:
df_NVDA

,date,open,high,low,close,volume,average,barCount
0,2026-01-30 09:30:00-05:00,6.13,6.17,6.02,6.04,-1.0,-1.0,-1
1,2026-01-30 10:00:00-05:00,6.04,6.15,6.04,6.11,-1.0,-1.0,-1
2,2026-01-30 11:00:00-05:00,6.11,6.11,6.03,6.06,-1.0,-1.0,-1
3,2026-01-30 12:00:00-05:00,6.06,6.09,6.04,6.08,-1.0,-1.0,-1
4,2026-01-30 13:00:00-05:00,6.08,6.10,6.05,6.06,-1.0,-1.0,-1
...,...,...,...,...,...,...,...,...
205,2026-03-13 11:00:00-04:00,6.07,6.10,6.01,6.06,-1.0,-1.0,-1
206,2026-03-13 12:00:00-04:00,6.06,6.08,6.00,6.04,-1.0,-1.0,-1
207,2026-03-13 13:00:00-04:00,6.04,6.09,6.02,6.04,-1.0,-1.0,-1
208,2026-03-13 14:00:00-04:00,6.04,6.08,6.02,6.08,-1.0,-1.0,-1


In [22]:
stock_COUR = Stock ('COUR', 'smart', 'USD')

bars = ib.reqHistoricalData(
    stock, endDateTime='', durationStr='30 D',
    barSizeSetting='1 hour', whatToShow='MIDPOINT', useRTH=True)

# convert to pandas dataframe (pandas needs to be installed):
df_COUR = util.df(bars)
print(df_COUR)

                         date  open  high   low  close  volume  average  \
0   2026-01-30 09:30:00-05:00  6.13  6.17  6.02   6.04    -1.0     -1.0   
1   2026-01-30 10:00:00-05:00  6.04  6.15  6.04   6.11    -1.0     -1.0   
2   2026-01-30 11:00:00-05:00  6.11  6.11  6.03   6.06    -1.0     -1.0   
3   2026-01-30 12:00:00-05:00  6.06  6.09  6.04   6.08    -1.0     -1.0   
4   2026-01-30 13:00:00-05:00  6.08  6.10  6.05   6.06    -1.0     -1.0   
..                        ...   ...   ...   ...    ...     ...      ...   
205 2026-03-13 11:00:00-04:00  6.07  6.10  6.01   6.06    -1.0     -1.0   
206 2026-03-13 12:00:00-04:00  6.06  6.08  6.00   6.04    -1.0     -1.0   
207 2026-03-13 13:00:00-04:00  6.04  6.09  6.02   6.04    -1.0     -1.0   
208 2026-03-13 14:00:00-04:00  6.04  6.08  6.02   6.08    -1.0     -1.0   
209 2026-03-13 15:00:00-04:00  6.08  6.09  6.01   6.04    -1.0     -1.0   

     barCount  
0          -1  
1          -1  
2          -1  
3          -1  
4          -1  
.. 

# 4. placing an order with account id 
    - Market
    -Limit

In [23]:
# When creating an order, you specify the account
my_order = MarketOrder('BUY', 100)
# my_order.account = 'DU6446107' #<-- Your actual account number goes here RK-paper trading
# my_order.account = 'DU6445903' #<-- Your actual account number goes here DAKK-paper trading
# my_order.account = 'XXX' Mot needed if connected through TWS

ib.placeOrder(stock_COUR, my_order)

Trade(contract=Stock(symbol='COUR', exchange='smart', currency='USD'), order=MarketOrder(orderId=8, clientId=3, action='BUY', totalQuantity=100), orderStatus=OrderStatus(orderId=8, status='PendingSubmit', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 15, 23, 39, 20, 971945, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0)], advancedError='')

In [24]:
my_order_limit = LimitOrder('BUY', 100, 5)
my_order_limit.account = 'DU6446107' #<-- Your actual account number goes here

ib.placeOrder(stock_COUR, my_order_limit)

Trade(contract=Stock(symbol='COUR', exchange='smart', currency='USD'), order=LimitOrder(orderId=9, clientId=3, action='BUY', totalQuantity=100, lmtPrice=5, account='DU6446107'), orderStatus=OrderStatus(orderId=9, status='PendingSubmit', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 15, 23, 39, 25, 661867, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0)], advancedError='')

# Error 
- below code did not work, due to xml or bs4. Print fundamentals resulted in blank output <[]>. Check later. worked in ibkr_v1 with in_insync

In [11]:
stock_AAPL = Stock('AAPL', 'SMART', 'USD')

# * 'ReportsFinSummary': Financial summary
# * 'ReportsOwnership': Company's ownership
# * 'ReportSnapshot': Company's financial overview
# * 'ReportsFinStatements': Financial Statements
# * 'RESC': Analyst Estimates
# * 'CalendarReport': Company's calendar

fundamentals = ib.reqFundamentalData(stock_AAPL, 'ReportSnapshot')

print(fundamentals)


# content = bs(fundamentals, "xml")

# ratios = content.find_all("Ratio")

# for ratio in ratios:
    # print(ratio['FieldName'])
    # print(ratio.text)



[]


In [13]:
x= 2+2



In [14]:
print(x)

4


# m